> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [9]:
# import zipfile
# import os
# import shutil
# import time

# # 1. 경로 설정
# dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
# extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

# !unzip /content/new_images_1000.zip -d /content/new_images_1000

# os.makedirs(extract_path, exist_ok=True)

# # 2. 메인 압축파일 해제
# print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
# with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# # 메인 압축파일 삭제 (요청사항)
# os.remove(dataset_zip)
# print("🗑️ 메인 압축파일 삭제 완료.")

# # 3. 내부 이미지 압축파일 통합 해제 로직
# print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
# for file in os.listdir(extract_path):
#     if file.endswith(".zip"):
#         file_path = os.path.join(extract_path, file)

#         # 이름 기반 대상 폴더 결정 (train/test 통합)
#         if 'train' in file.lower():
#             target_folder_name = "train_images"
#         elif 'test' in file.lower():
#             target_folder_name = "test_images"
#         else:
#             target_folder_name = file.replace(".zip", "")

#         target_subfolder = os.path.join(extract_path, target_folder_name)
#         os.makedirs(target_subfolder, exist_ok=True)

#         print(f"📂 {file} -> {target_folder_name} 통합 중...")

#         with zipfile.ZipFile(file_path, 'r') as zip_ref:
#             for member in zip_ref.infolist():
#                 if not member.is_dir():
#                     # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
#                     filename = os.path.basename(member.filename)
#                     if filename:
#                         target_file_path = os.path.join(target_subfolder, filename)
#                         with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
#                             shutil.copyfileobj(source, target)

#         # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
#         try:
#             time.sleep(0.5)
#             if os.path.exists(file_path):
#                 os.remove(file_path)
#                 print(f"🗑️ 삭제 성공: {file}")
#         except Exception as e:
#             print(f"❌ {file} 삭제 실패: {e}")

# print("\n✨ 모든 작업이 완료되었습니다!")
# print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

In [ ]:
import os

if not os.path.exists("/content/new_images_1000"):
    !unzip /content/new_images_1000.zip -d /content/new_images_1000

if not os.path.exists("/content/TL_81_단일"):
    !unzip /content/TL_81_단일.zip -d /content/TL_81_단일

- 구글 드라이브 휴지통 비우기

In [10]:
from google.colab import auth
from googleapiclient.discovery import build

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 휴지통 완전히 비우기 함수
def empty_trash():
    try:
        drive_service.files().emptyTrash().execute()
        print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
    except Exception as e:
        print(f"❌ 휴지통 비우기 실패: {e}")

# 실행
empty_trash()

✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.


> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [11]:
import sys
!{sys.executable} -m pip install -q ultralytics pycocotools pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.5 MB/s eta 0:00:00


In [14]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################

import os
import json
import yaml
import shutil
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval



############################################################
# 0. 경로 / 설정
############################################################

def normalize_path(path):
    return unicodedata.normalize("NFC", path).strip()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset")

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH  = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR   = os.path.join(extract_path, "train_images")
TEST_IMG_DIR    = os.path.join(extract_path, "test_images")
EXTRA_IMG_DIR   = "/content/new_images_1000"

WORK_DIR   = os.path.join(extract_path, "yolo_effb3_pipeline")
YOLO_ROOT  = os.path.join(WORK_DIR, "yolo_oneclass")
MODEL_DIR  = os.path.join(WORK_DIR, "saved_models")
PRED_DIR   = os.path.join(WORK_DIR, "predictions")
EVAL_DIR   = os.path.join(WORK_DIR, "eval")

ensure_dir(WORK_DIR)
ensure_dir(YOLO_ROOT)
ensure_dir(MODEL_DIR)
ensure_dir(PRED_DIR)
ensure_dir(EVAL_DIR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"

# 네 환경에 맞게 수정
YOLO_MODEL_NAME = "/content/drive/MyDrive/data/초급_프로젝트/dataset/models/yolo26m.pt"   # 예: "/content/drive/MyDrive/weights/yolo26m.pt"

# detector
YOLO_IMGSZ = 640
YOLO_EPOCHS = 30
YOLO_BATCH = 8
YOLO_WORKERS = 0   # 중요: Colab thread 에러 방지

# classifier
CLS_IMG_SIZE = 300
CLS_EPOCHS = 10
CLS_BATCH = 16
CLS_LR = 3e-4
CLS_NUM_WORKERS = 0   # 중요

# crop
CROP_PAD_RATIO = 0.08

# 추론 / 제출
DET_CONF = 0.05
DET_IOU = 0.5
MAX_SUB_PER_IMAGE = 4
SUBMISSION_CATEGORY_OFFSET = 1   # 제출 사이트가 +1 요구하면 1, 아니면 0

print("DEVICE:", DEVICE)
print("TRAIN_JSON_PATH exists:", os.path.exists(TRAIN_JSON_PATH))
print("TEST_JSON_PATH exists :", os.path.exists(TEST_JSON_PATH))
print("TRAIN_IMG_DIR exists  :", os.path.exists(TRAIN_IMG_DIR))
print("TEST_IMG_DIR exists   :", os.path.exists(TEST_IMG_DIR))

############################################################
# 1. COCO JSON -> images_df / ann_df
############################################################

def load_coco_tables(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_columns = [
        "coco_image_id", "file_name", "image_stem",
        "image_path", "img_w", "img_h"
    ]

    ann_columns = [
        "ann_id", "coco_image_id", "file_name", "image_stem", "image_path",
        "img_w", "img_h", "category_id",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h"
    ]

    image_records = []
    image_id_to_info = {}

    for img in data.get("images", []):
        coco_image_id = int(img["id"])
        file_name = img["file_name"]
        image_path = os.path.join(img_dir, file_name)

        if not os.path.exists(image_path):
            continue

        width = img.get("width", None)
        height = img.get("height", None)

        if width is None or height is None:
            with Image.open(image_path) as im:
                width, height = im.size

        rec = {
            "coco_image_id": coco_image_id,
            "file_name": file_name,
            "image_stem": os.path.splitext(file_name)[0],
            "image_path": image_path,
            "img_w": int(width),
            "img_h": int(height),
        }
        image_records.append(rec)
        image_id_to_info[coco_image_id] = rec

    images_df = pd.DataFrame(image_records, columns=image_columns)

    ann_records = []
    for ann in data.get("annotations", []):
        coco_image_id = int(ann["image_id"])

        if coco_image_id not in image_id_to_info:
            continue

        bbox = ann.get("bbox", None)
        if bbox is None or len(bbox) != 4:
            continue

        x, y, w, h = bbox
        info = image_id_to_info[coco_image_id]

        ann_records.append({
            "ann_id": int(ann.get("id", len(ann_records) + 1)),
            "coco_image_id": coco_image_id,
            "file_name": info["file_name"],
            "image_stem": info["image_stem"],
            "image_path": info["image_path"],
            "img_w": info["img_w"],
            "img_h": info["img_h"],
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    ann_df = pd.DataFrame(ann_records, columns=ann_columns)
    return images_df, ann_df, data



train_images_df, train_ann_df, train_raw_json = load_coco_tables(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
test_images_df, test_ann_df, test_raw_json = load_coco_tables(TEST_JSON_PATH, TEST_IMG_DIR)

print("train image 수 :", len(train_images_df))
print("train ann 수   :", len(train_ann_df))
print("test image 수  :", len(test_images_df))
print("test ann 수    :", len(test_ann_df))
print(train_ann_df.head())

############################################################
# 추가 이미지 병합 (중복 파일 스킵)
############################################################

extra_image_records = []
existing_stems = set(train_images_df["image_stem"].tolist())

for img_fname in os.listdir(EXTRA_IMG_DIR):
    if not img_fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    stem = os.path.splitext(img_fname)[0]

    if stem in existing_stems:
        print(f"중복 스킵: {img_fname}")
        continue

    img_path = os.path.join(EXTRA_IMG_DIR, img_fname)
    with Image.open(img_path) as im:
        w, h = im.size

    extra_image_records.append({
        "coco_image_id": -1,
        "file_name": img_fname,
        "image_stem": stem,
        "image_path": img_path,
        "img_w": int(w),
        "img_h": int(h),
    })
    existing_stems.add(stem)

extra_images_df = pd.DataFrame(extra_image_records)
train_images_df = pd.concat([train_images_df, extra_images_df], ignore_index=True)

print(f"추가 이미지 {len(extra_image_records)}장 병합 완료")
print(f"[최종] train image 수: {len(train_images_df)}")

############################################################
# 추가 이미지 라벨 로드 (JSON -> ann_df 병합)
############################################################

EXTRA_JSON_DIR = "/content/TL_81_단일/TL_81_단일"
extra_ann_records = []
max_ann_id = int(train_ann_df["ann_id"].max()) if len(train_ann_df) > 0 else 0

for img_fname in extra_image_records:
    stem = img_fname["image_stem"]
    kid = stem.split('_')[0]
    json_path = os.path.join(EXTRA_JSON_DIR, kid + "_json", stem + ".json")

    if not os.path.exists(json_path):
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    img_info = data["images"][0]
    W, H = img_info["width"], img_info["height"]

    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        max_ann_id += 1
        extra_ann_records.append({
            "ann_id": max_ann_id,
            "coco_image_id": -1,
            "file_name": img_fname["file_name"],
            "image_stem": stem,
            "image_path": img_fname["image_path"],
            "img_w": int(W),
            "img_h": int(H),
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

extra_ann_df = pd.DataFrame(extra_ann_records)
train_ann_df = pd.concat([train_ann_df, extra_ann_df], ignore_index=True)
print(f"추가 라벨 {len(extra_ann_records)}개 병합 완료")
print(f"[최종] train ann 수: {len(train_ann_df)}")

############################################################
# 2. bbox 정리 / split / category mapping
############################################################

def sanitize_ann_df(ann_df):
    bad_cols = [
        "ann_id", "file_name", "image_stem",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h", "img_w", "img_h"
    ]

    if ann_df is None or len(ann_df) == 0:
        return ann_df.copy(), pd.DataFrame(columns=bad_cols)

    required_cols = ["bbox_x", "bbox_y", "bbox_w", "bbox_h", "img_w", "img_h"]
    missing = [c for c in required_cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"sanitize_ann_df에 필요한 컬럼이 없습니다: {missing}")

    df = ann_df.copy()
    for c in required_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    x1 = df["bbox_x"].to_numpy(dtype=float)
    y1 = df["bbox_y"].to_numpy(dtype=float)
    x2 = x1 + df["bbox_w"].to_numpy(dtype=float)
    y2 = y1 + df["bbox_h"].to_numpy(dtype=float)

    x1n = np.minimum(x1, x2)
    y1n = np.minimum(y1, y2)
    x2n = np.maximum(x1, x2)
    y2n = np.maximum(y1, y2)

    img_w = df["img_w"].to_numpy(dtype=float)
    img_h = df["img_h"].to_numpy(dtype=float)

    x1c = np.clip(x1n, 0, img_w)
    y1c = np.clip(y1n, 0, img_h)
    x2c = np.clip(x2n, 0, img_w)
    y2c = np.clip(y2n, 0, img_h)

    valid = (
        np.isfinite(x1c) & np.isfinite(y1c) &
        np.isfinite(x2c) & np.isfinite(y2c) &
        np.isfinite(img_w) & np.isfinite(img_h) &
        (x2c > x1c) & (y2c > y1c)
    )

    bad_df = df.loc[~valid, bad_cols].copy()

    clean_df = df.loc[valid].copy()
    clean_df["bbox_x"] = x1c[valid]
    clean_df["bbox_y"] = y1c[valid]
    clean_df["bbox_w"] = x2c[valid] - x1c[valid]
    clean_df["bbox_h"] = y2c[valid] - y1c[valid]

    return clean_df.reset_index(drop=True), bad_df.reset_index(drop=True)


train_ann_df, bad_train_ann_df = sanitize_ann_df(train_ann_df)

if len(test_ann_df) > 0:
    test_ann_df, bad_test_ann_df = sanitize_ann_df(test_ann_df)
else:
    bad_test_ann_df = pd.DataFrame()

print("정리 후 train ann 수:", len(train_ann_df))
print("제거된 잘못된 train ann 수:", len(bad_train_ann_df))
print("정리 후 test ann 수 :", len(test_ann_df))


DEVICE: cuda
TRAIN_JSON_PATH exists: True
TEST_JSON_PATH exists : True
TRAIN_IMG_DIR exists  : True
TEST_IMG_DIR exists   : True
train image 수 : 1489
train ann 수   : 4526
test image 수  : 0
test ann 수    : 0
   ann_id  coco_image_id                                          file_name  \
0       1              1  K-001900-016551-024850-027926_0_2_0_2_90_000_2...   
1       2              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   
2       3              3  K-001900-016551-024850-027926_0_2_0_2_75_000_2...   
3       6              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   
4       7              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   

                                         image_stem  \
0  K-001900-016551-024850-027926_0_2_0_2_90_000_200   
1  K-001900-016551-024850-027926_0_2_0_2_70_000_200   
2  K-001900-016551-024850-027926_0_2_0_2_75_000_200   
3  K-001900-016551-024850-027926_0_2_0_2_70_000_200   
4  K-001900-016551-024850-027926_0_2_0_2_70

In [15]:
############################################################
# 2. train/val split + classifier용 category 매핑
############################################################

def split_images(images_df, train_ratio=0.9, seed=42):
    stems = images_df["image_stem"].tolist()
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(stems), generator=g).tolist()
    split = int(len(stems) * train_ratio)

    train_stems = {stems[i] for i in perm[:split]}
    val_stems   = {stems[i] for i in perm[split:]}
    return train_stems, val_stems

def ensure_all_classes_in_train(ann_df, train_stems, val_stems):
    train_stems = set(train_stems)
    val_stems = set(val_stems)

    cats = sorted(ann_df["category_id"].astype(int).unique().tolist())

    for cat in cats:
        in_train = ann_df[
            (ann_df["image_stem"].isin(train_stems)) &
            (ann_df["category_id"].astype(int) == cat)
        ]
        if len(in_train) == 0:
            candidates = ann_df[
                (ann_df["image_stem"].isin(val_stems)) &
                (ann_df["category_id"].astype(int) == cat)
            ]["image_stem"].drop_duplicates().tolist()

            if len(candidates) > 0:
                chosen = candidates[0]
                val_stems.remove(chosen)
                train_stems.add(chosen)
                print(f"[split 보정] val -> train 이동: {chosen} (category={cat})")

    return train_stems, val_stems


train_stems, val_stems = split_images(train_images_df, train_ratio=0.9, seed=42)
train_stems, val_stems = ensure_all_classes_in_train(train_ann_df, train_stems, val_stems)

print("train image split:", len(train_stems))
print("val image split  :", len(val_stems))


# classifier용 category mapping
unique_cats = sorted(train_ann_df["category_id"].astype(int).unique().tolist())
orig2cls = {orig_cat: idx for idx, orig_cat in enumerate(unique_cats)}
cls2orig = {idx: orig_cat for orig_cat, idx in orig2cls.items()}

CLS_LABEL_MAP_PATH = os.path.join(MODEL_DIR, "cls_idx_to_orig_cat.json")
with open(CLS_LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({str(k): int(v) for k, v in cls2orig.items()}, f, ensure_ascii=False, indent=2)

print("분류 클래스 수:", len(unique_cats))
print("label map 저장:", CLS_LABEL_MAP_PATH)



train image split: 1340
val image split  : 149
분류 클래스 수: 73
label map 저장: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/saved_models/cls_idx_to_orig_cat.json


In [16]:
############################################################
# 3. COCO -> YOLO one-class dataset export
############################################################

def copy_if_needed(src, dst):
    ensure_dir(os.path.dirname(dst))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def coco_xywh_to_yolo(x, y, w, h, img_w, img_h):
    xc = (x + w / 2.0) / img_w
    yc = (y + h / 2.0) / img_h
    wn = w / img_w
    hn = h / img_h
    return xc, yc, wn, hn

def export_yolo_oneclass_dataset(images_df, ann_df, train_stems, val_stems, out_root):
    for split_name in ["train", "val"]:
        ensure_dir(os.path.join(out_root, "images", split_name))
        ensure_dir(os.path.join(out_root, "labels", split_name))

    ann_groups = {stem: g.copy() for stem, g in ann_df.groupby("image_stem")}
    split_map = {"train": train_stems, "val": val_stems}

    for split_name, stems in split_map.items():
        sub_images = images_df[images_df["image_stem"].isin(stems)].copy()

        for _, row in tqdm(sub_images.iterrows(), total=len(sub_images), desc=f"YOLO export [{split_name}]"):
            image_stem = row["image_stem"]
            src_img = row["image_path"]
            dst_img = os.path.join(out_root, "images", split_name, row["file_name"])
            copy_if_needed(src_img, dst_img)

            label_path = os.path.join(out_root, "labels", split_name, f"{image_stem}.txt")
            img_w, img_h = int(row["img_w"]), int(row["img_h"])
            anns = ann_groups.get(image_stem, None)

            with open(label_path, "w", encoding="utf-8") as f:
                if anns is not None and len(anns) > 0:
                    for _, ann_row in anns.iterrows():
                        x, y, w, h = ann_row["bbox_x"], ann_row["bbox_y"], ann_row["bbox_w"], ann_row["bbox_h"]
                        xc, yc, wn, hn = coco_xywh_to_yolo(x, y, w, h, img_w, img_h)
                        f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    data_yaml = {
        "path": out_root,
        "train": "images/train",
        "val": "images/val",
        "names": ["drug"],
    }

    yaml_path = os.path.join(out_root, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    return yaml_path


YOLO_DATA_YAML = export_yolo_oneclass_dataset(
    images_df=train_images_df,
    ann_df=train_ann_df,
    train_stems=train_stems,
    val_stems=val_stems,
    out_root=YOLO_ROOT
)

print("YOLO data yaml:", YOLO_DATA_YAML)


############################################################
# 4. EfficientNet-B3 classifier Dataset / DataLoader
############################################################

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

cls_train_tfms = T.Compose([
    T.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cls_val_tfms = T.Compose([
    T.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def safe_crop_xywh(image, x, y, w, h, pad_ratio=0.08):
    img_w, img_h = image.size

    x = float(x); y = float(y); w = float(w); h = float(h)
    if w <= 0 or h <= 0:
        return None

    x1 = x
    y1 = y
    x2 = x + w
    y2 = y + h

    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1

    pad_x = (x2 - x1) * pad_ratio
    pad_y = (y2 - y1) * pad_ratio

    left   = int(np.floor(x1 - pad_x))
    top    = int(np.floor(y1 - pad_y))
    right  = int(np.ceil(x2 + pad_x))
    bottom = int(np.ceil(y2 + pad_y))

    left   = max(0, min(left, img_w))
    top    = max(0, min(top, img_h))
    right  = max(0, min(right, img_w))
    bottom = max(0, min(bottom, img_h))

    if right <= left or bottom <= top:
        return None

    return image.crop((left, top, right, bottom))

class CropClassificationDataset(Dataset):
    def __init__(self, ann_df, stems, orig2cls, transform=None, pad_ratio=0.08):
        self.df = ann_df[ann_df["image_stem"].isin(stems)].reset_index(drop=True).copy()
        self.orig2cls = orig2cls
        self.transform = transform
        self.pad_ratio = pad_ratio

        # 혹시 category가 mapping 밖이면 제거
        self.df = self.df[self.df["category_id"].astype(int).isin(self.orig2cls.keys())].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        with Image.open(img_path).convert("RGB") as image:
            crop = safe_crop_xywh(
                image=image,
                x=row["bbox_x"],
                y=row["bbox_y"],
                w=row["bbox_w"],
                h=row["bbox_h"],
                pad_ratio=self.pad_ratio
            )

            if crop is None:
                crop = image.copy()

        label = self.orig2cls[int(row["category_id"])]

        if self.transform is not None:
            crop = self.transform(crop)

        return crop, label


train_cls_ds = CropClassificationDataset(
    ann_df=train_ann_df,
    stems=train_stems,
    orig2cls=orig2cls,
    transform=cls_train_tfms,
    pad_ratio=CROP_PAD_RATIO
)

val_cls_ds = CropClassificationDataset(
    ann_df=train_ann_df,
    stems=val_stems,
    orig2cls=orig2cls,
    transform=cls_val_tfms,
    pad_ratio=CROP_PAD_RATIO
)

train_cls_loader = DataLoader(
    train_cls_ds,
    batch_size=CLS_BATCH,
    shuffle=True,
    num_workers=CLS_NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_cls_loader = DataLoader(
    val_cls_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=CLS_NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("classifier train size:", len(train_cls_ds))
print("classifier val size  :", len(val_cls_ds))

YOLO export [train]:   0%|          | 0/1340 [00:00<?, ?it/s]

YOLO export [val]:   0%|          | 0/149 [00:00<?, ?it/s]

YOLO data yaml: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo_oneclass/data.yaml
classifier train size: 4079
classifier val size  : 445


In [17]:
############################################################
# 5-1. YOLOv26m one-class detector 학습
############################################################

det_model = YOLO(YOLO_MODEL_NAME)

det_model.train(
    data=YOLO_DATA_YAML,
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    device=YOLO_DEVICE,
    project=WORK_DIR,
    name="yolo26m_oneclass",
    exist_ok=True,
    workers=YOLO_WORKERS,
    patience=10,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

############################################################
# 5-2. YOLO best weight 경로 찾기
############################################################

def resolve_yolo_best_path(work_dir, run_name="yolo26m_oneclass"):
    candidates = [
        os.path.join(work_dir, run_name, "weights", "best.pt"),
        os.path.join(work_dir, run_name, "weights", "last.pt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"YOLO weight를 찾지 못했습니다. candidates={candidates}")

YOLO_BEST_PT = resolve_yolo_best_path(WORK_DIR, "yolo26m_oneclass")
print("YOLO_BEST_PT:", YOLO_BEST_PT)

############################################################
# 5-3. detector validation metric (선택)
############################################################

detector_for_val = YOLO(YOLO_BEST_PT)

det_metrics = detector_for_val.val(
    data=YOLO_DATA_YAML,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    device=YOLO_DEVICE,
    split="val",
    workers=YOLO_WORKERS,
    verbose=False
)

print("YOLO one-class detector 결과")
print("mAP50-95:", det_metrics.box.map)
print("mAP50   :", det_metrics.box.map50)
print("mAP75   :", det_metrics.box.map75)


############################################################
# 6-1. EfficientNet-B3 classifier 설정
############################################################

cls_model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

in_features = cls_model.classifier[1].in_features
cls_model.classifier[1] = nn.Linear(in_features, len(unique_cats))
cls_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(cls_model.parameters(), lr=CLS_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CLS_EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

CLS_BEST_PTH = os.path.join(MODEL_DIR, "efficientnet_b3_cls_best.pth")
best_val_acc = 0.0

print("classifier 준비 완료")
print("num_classes:", len(unique_cats))
print("save path:", CLS_BEST_PTH)

############################################################
# 6-2. EfficientNet-B3 classifier 학습
############################################################

for epoch in range(CLS_EPOCHS):
    # train
    cls_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_cls_loader, desc=f"CLS Train [{epoch+1}/{CLS_EPOCHS}]"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = cls_model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / max(1, train_total)
    train_acc = train_correct / max(1, train_total)

    # val
    cls_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_cls_loader, desc=f"CLS Val [{epoch+1}/{CLS_EPOCHS}]"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = cls_model(images)
                loss = criterion(logits, labels)

            val_loss_sum += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / max(1, val_total)
    val_acc = val_correct / max(1, val_total)

    scheduler.step()

    print(
        f"[Epoch {epoch+1}/{CLS_EPOCHS}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(cls_model.state_dict(), CLS_BEST_PTH)
        print("✅ classifier best 저장:", CLS_BEST_PTH)

print("classifier 학습 완료")



Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo_oneclass/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/drive/MyDrive/data/초급_프로젝트/dataset/models/yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_o

100%|██████████| 47.2M/47.2M [00:00<00:00, 196MB/s]


classifier 준비 완료
num_classes: 73
save path: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/saved_models/efficientnet_b3_cls_best.pth


/tmp/ipykernel_1477/2908200399.py:84: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


CLS Train [1/10]:   0%|          | 0/255 [00:00<?, ?it/s]

...

CLS Val [9/10]:   0%|          | 0/28 [00:00<?, ?it/s]

[Epoch 9/10] train_loss=0.4310, train_acc=0.9973 | val_loss=0.4507, val_acc=0.9955


CLS Train [10/10]:   0%|          | 0/255 [00:00<?, ?it/s]

CLS Val [10/10]:   0%|          | 0/28 [00:00<?, ?it/s]

[Epoch 10/10] train_loss=0.4281, train_acc=0.9978 | val_loss=0.4493, val_acc=0.9955
classifier 학습 완료


In [20]:
############################################################
# 7. test 추론 -> submission 생성
############################################################

# detector load
detector = YOLO(YOLO_BEST_PT)

# classifier load
with open(CLS_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    cls2orig_loaded = {int(k): int(v) for k, v in json.load(f).items()}

classifier = efficientnet_b3(weights=None)
in_features = classifier.classifier[1].in_features
classifier.classifier[1] = nn.Linear(in_features, len(cls2orig_loaded))
classifier.load_state_dict(torch.load(CLS_BEST_PTH, map_location=DEVICE))
classifier.to(DEVICE)
classifier.eval()


@torch.no_grad()
def classify_crop(pil_crop, classifier, transform, device):
    x = transform(pil_crop).unsqueeze(0).to(device)
    logits = classifier(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).item())
    pred_conf = float(probs[pred_idx].item())
    return pred_idx, pred_conf

def run_pipeline_on_image(
    img_path,
    detector,
    classifier,
    cls_transform,
    cls2orig,
    device,
    conf=0.05,
    iou=0.5,
    imgsz=640,
    yolo_device=0,
    crop_pad_ratio=0.08
):
    image = Image.open(img_path).convert("RGB")

    results = detector.predict(
        source=img_path,
        imgsz=imgsz,
        conf=conf,
        iou=iou,
        max_det=100,
        device=yolo_device,
        verbose=False
    )

    result = results[0]
    outputs = []

    if result.boxes is None or len(result.boxes) == 0:
        return outputs

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    det_scores = result.boxes.conf.cpu().numpy()

    for box, det_score in zip(boxes_xyxy, det_scores):
        x1, y1, x2, y2 = map(float, box)
        w = x2 - x1
        h = y2 - y1

        crop = safe_crop_xywh(image, x1, y1, w, h, pad_ratio=crop_pad_ratio)
        if crop is None:
            continue

        pred_idx, cls_score = classify_crop(crop, classifier, cls_transform, device)
        orig_cat = cls2orig[pred_idx]

        outputs.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": w, "h": h,
            "det_score": float(det_score),
            "cls_idx": int(pred_idx),
            "orig_cat": int(orig_cat),
            "cls_score": float(cls_score),
            "final_score": float(det_score * cls_score),
        })

    return outputs


test_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp"))
])

rows_submit = []
annotation_id = 1

for f in tqdm(test_files, desc="Test inference"):
    img_path = os.path.join(TEST_IMG_DIR, f)
    image_stem = os.path.splitext(f)[0]

    preds = run_pipeline_on_image(
        img_path=img_path,
        detector=detector,
        classifier=classifier,
        cls_transform=cls_val_tfms,
        cls2orig=cls2orig_loaded,
        device=DEVICE,
        conf=DET_CONF,
        iou=DET_IOU,
        imgsz=YOLO_IMGSZ,
        yolo_device=YOLO_DEVICE,
        crop_pad_ratio=CROP_PAD_RATIO
    )

    preds = sorted(preds, key=lambda x: x["final_score"], reverse=True)[:MAX_SUB_PER_IMAGE]

    for p in preds:
        submit_cat = int(p["orig_cat"]) + SUBMISSION_CATEGORY_OFFSET

        rows_submit.append({
            "annotation_id": annotation_id,
            "image_id": image_stem,   # 제출 규칙에 맞게 유지
            "category_id": submit_cat,
            "bbox_x": p["x1"],
            "bbox_y": p["y1"],
            "bbox_w": p["w"],
            "bbox_h": p["h"],
            "score": p["final_score"],
        })
        annotation_id += 1

df_sub = pd.DataFrame(rows_submit, columns=[
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])

SUBMISSION_CSV_PATH = os.path.join(PRED_DIR, "final_submission_yolo26m_effb3.csv")
df_sub.to_csv(SUBMISSION_CSV_PATH, index=False)

print("✅ submission 저장:", SUBMISSION_CSV_PATH)
print("총 제출 row 수:", len(df_sub))
print(df_sub.head())



Test inference:   0%|          | 0/843 [00:00<?, ?it/s]

✅ submission 저장: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/predictions/final_submission_yolo26m_effb3.csv
총 제출 row 수: 3254
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        16551  555.805481   70.166534  396.709778   
1              2        1         1900  157.162003  252.611816  205.644974   
2              3        1        27926  597.864258  673.938477  261.468079   
3              4        1        24850  173.969864  743.866333  180.675034   
4              5       10        16548  102.183846  808.173950  241.217369   

       bbox_h     score  
0  407.726074  0.744943  
1  123.204590  0.740137  
2  479.743164  0.702337  
3  289.760010  0.701345  
4  237.470703  0.777453  


In [21]:
############################################################
# 8. val split 기준 최종 파이프라인 성능 평가 (COCO mAP)
############################################################

def build_val_coco_gt(train_raw_json, val_stems):
    val_stems = set(val_stems)

    images = []
    annotations = []
    valid_image_ids = set()

    for img in train_raw_json.get("images", []):
        stem = os.path.splitext(img["file_name"])[0]
        if stem in val_stems:
            images.append(img)
            valid_image_ids.add(int(img["id"]))

    for ann in train_raw_json.get("annotations", []):
        if int(ann["image_id"]) in valid_image_ids:
            annotations.append(ann)

    val_gt = {
        "images": images,
        "annotations": annotations,
        "categories": train_raw_json.get("categories", [])
    }

    return val_gt
VAL_GT_JSON_PATH = os.path.join(EVAL_DIR, "val_gt.json")
val_gt = build_val_coco_gt(train_raw_json, val_stems)

with open(VAL_GT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(val_gt, f, ensure_ascii=False)

print("VAL_GT_JSON_PATH:", VAL_GT_JSON_PATH)
print("val images:", len(val_gt["images"]))
print("val anns  :", len(val_gt["annotations"]))
# val image stem -> coco image id
val_stem_to_gtid = {
    os.path.splitext(img["file_name"])[0]: int(img["id"])
    for img in val_gt["images"]
}

val_image_rows = train_images_df[train_images_df["image_stem"].isin(val_stems)].copy()

rows_eval = []

for _, row in tqdm(val_image_rows.iterrows(), total=len(val_image_rows), desc="Val pipeline inference"):
    img_path = row["image_path"]
    image_stem = row["image_stem"]

    preds = run_pipeline_on_image(
        img_path=img_path,
        detector=detector,
        classifier=classifier,
        cls_transform=cls_val_tfms,
        cls2orig=cls2orig_loaded,
        device=DEVICE,
        conf=DET_CONF,
        iou=DET_IOU,
        imgsz=YOLO_IMGSZ,
        yolo_device=YOLO_DEVICE,
        crop_pad_ratio=CROP_PAD_RATIO
    )

    for p in preds:
        rows_eval.append({
            "image_id": val_stem_to_gtid[image_stem],
            "category_id": int(p["orig_cat"]),  # 로컬 eval은 원본 category
            "bbox": [p["x1"], p["y1"], p["w"], p["h"]],
            "score": float(p["final_score"]),
        })

VAL_PRED_JSON_PATH = os.path.join(EVAL_DIR, "val_predictions_yolo26m_effb3.json")
with open(VAL_PRED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(rows_eval, f, ensure_ascii=False)

print("VAL_PRED_JSON_PATH:", VAL_PRED_JSON_PATH)
print("val prediction 수:", len(rows_eval))
coco_gt = COCO(VAL_GT_JSON_PATH)
coco_dt = coco_gt.loadRes(VAL_PRED_JSON_PATH)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

VAL_GT_JSON_PATH: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/eval/val_gt.json
val images: 149
val anns  : 445


Val pipeline inference:   0%|          | 0/149 [00:00<?, ?it/s]

VAL_PRED_JSON_PATH: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/eval/val_predictions_yolo26m_effb3.json
val prediction 수: 603
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.30s).
Accumulating evaluation results...
DONE (t=0.56s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.887
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.902
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.902
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.887
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.9


결론부터 말씀드리면, 현재 모델은 **"정답은 기가 막히게 잘 찾아내지만(Recall), 위치를 정밀하게 그리는 능력(Precision)은 아직 보완이 필요하다"** 고 요약할 수 있습니다.

<br>

### **📊 핵심 지표 분석 (Performance Review)**

#### **1. Precision vs Recall의 불균형**

* **mAP (IoU 0.50:0.95):** **0.374**
* **mAR (maxDets 100):** **0.898**

* **해석:** 리콜(Recall)이 **0.898**로 매우 높습니다. 이는 모델이 이미지 속 물체를 **거의 놓치지 않고 다 찾아내고 있다**는 뜻입니다. 반면 AP는 그에 비해 낮습니다. 즉, 물체를 찾긴 찾는데 **Bounding Box의 위치가 약간 어긋나 있거나, 오탐(False Positive)이 섞여 있을 확률**이 높습니다.

<br>

#### **2. IoU 임계값(Threshold)에 따른 변화**

* **AP @ 0.50:** **0.397**
* **AP @ 0.75:** **0.395**

    * **해석:** 보통 IoU 기준이 엄격해질수록(0.5 $\rightarrow$ 0.75) 점수가 팍 떨어지기 마련인데, 이 모델은 거의 차이가 없습니다.
    * **진단:** 이는 모델이 **Box를 아주 대충 그리거나, 아주 정확하게 그리거나 둘 중 하나**라는 뜻입니다. 어중간하게 틀리는 게 아니라 위치 정확도 면에서 어떤 일관된 특징(예: 항상 실제보다 조금 크게 그림)이 있을 수 있습니다.

<br>

#### **3. Object Size에 따른 성능 (중요!)**

* **Small / Medium:** **-1.000**
* **Large:** **0.374**

    * **해석:** `-1.000`은 해당 데이터가 **평가셋에 아예 존재하지 않는다**는 뜻입니다.
    * **진단:** 현재 재찬 님이 다루시는 데이터셋의 물체들은 모두 **'Large'** 카테고리에 속합니다. 작은 물체를 탐지할 걱정은 안 해도 되지만, 큰 물체의 전체적인 윤곽을 더 정밀하게 잡는 데 집중해야 합니다.

<br>

### **💡 1타 강사의 '성능 향상' 클리닉**

재찬 님, 점수를 더 올리고 싶다면 수강생들에게 다음 **3가지 튜닝 전략**을 가르쳐주며 적용해 보세요.

1. **Localization Loss 강화:** 현재 리콜은 충분하니, 박스 위치를 더 정확히 잡아야 합니다. `CIoU`나 `DIoU` 같은 Loss 함수를 쓰고 있는지 확인하고, Box Regression의 가중치를 조금 높여보세요.

2. **Confidence Threshold 튜닝:** 리콜이 0.9에 가깝다는 건 모델이 아주 자신 있게(혹은 너무 남발해서) 박스를 치고 있다는 겁니다. Confidence 임계값을 살짝 높여서 확실한 것만 남기면 AP(정밀도)가 올라갈 수 있습니다.

3. **데이터 증강(Augmentation) 점검:** 물체가 크기 때문에(Large), 이미지 가장자리에 걸린 물체를 잘 잡는지 확인해 보세요. `Random Crop` 보다는 `Scaling`이나 `Translation` 위주의 증강이 더 효과적일 수 있습니다.

<br>

### **🎯 최종 리포트**

> **"모델이 눈은 좋은데(Recall), 손재주가 살짝 부족합니다(Precision). 박스를 더 예쁘게 그리도록 가르치면 점수는 수직 상승할 겁니다!"**